In [55]:
import sys
sys.path.append("../")

import sympy as sp
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir
from affine_body_core import compute_J_point, compute_J_vec

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")
Gen.DisableLatexComment()

In [56]:
kappa  = Eigen.Scalar("kappa")

# Basis vectors
ci_bar = Eigen.Vector("ci_bar",3)
ti_bar = Eigen.Vector("ti_bar",3)
ni_bar = Eigen.Vector("ni_bar",3)
bi_bar = Eigen.Vector("bi_bar",3)

cj_bar = Eigen.Vector("cj_bar",3)
tj_bar = Eigen.Vector("tj_bar",3)
nj_bar = Eigen.Vector("nj_bar",3)
bj_bar = Eigen.Vector("bj_bar",3)

# Affine Body DOF vectors
qi = Eigen.Vector("qi",12)
qj = Eigen.Vector("qj",12)

$$
\begin{aligned}
C_0 &= \hat{\mathbf{t}}_i - \hat{\mathbf{t}}_j &= 0 \\
C_1 &= \hat{\mathbf{n}}_i - \hat{\mathbf{n}}_j &= 0 \\
C_2 &= \hat{\mathbf{b}}_i - \hat{\mathbf{b}}_j &= 0 \\
\end{aligned}
$$

$$
\mathbf{F}_{r} = \begin{bmatrix}
\mathbf{t}_i - \mathbf{t}_j\\
\mathbf{n}_i - \mathbf{n}_j\\
\mathbf{b}_i - \mathbf{b}_j\\
\end{bmatrix}_{9 \times 1}
$$

Frame Affine Body Mapping:

$$
\mathbf{J}_r = \begin{bmatrix}
\mathring{\mathbf{J}}(\bar{\mathbf{t}}_i) & \mathring{-\mathbf{J}}(\bar{\mathbf{t}}_j) \\
\mathring{\mathbf{J}}(\bar{\mathbf{n}}_i) & \mathring{-\mathbf{J}}(\bar{\mathbf{n}}_j) \\
\mathring{\mathbf{J}}(\bar{\mathbf{b}}_i) & \mathring{-\mathbf{J}}(\bar{\mathbf{b}}_j)  \\
\end{bmatrix}_{9 \times 24}
$$

$$
\mathbf{F}_{r} = \mathbf{J}_{r} \cdot
\begin{bmatrix}
\mathbf{q}_i \\
\mathbf{q}_j \\
\end{bmatrix}
$$

In [57]:
# Mapping
Jr = sp.Matrix.zeros(9,24)
Jr[0:3, 0:12] = compute_J_vec(ti_bar)
Jr[0:3, 12:24] = -compute_J_vec(tj_bar)
Jr[3:6, 0:12] = compute_J_vec(ni_bar)
Jr[3:6, 12:24] = -compute_J_vec(nj_bar)
Jr[6:9, 0:12] = compute_J_vec(bi_bar)
Jr[6:9, 12:24] = -compute_J_vec(bj_bar)
content = ""

# from ABD q to F
Fr_q = Jr @ sp.Matrix.vstack(qi, qj)
Cl_Fr = Gen.Closure(ti_bar, ni_bar, bi_bar, qi, # Affine Body i
                     tj_bar, nj_bar, bj_bar, qj) # Affine Body j
# from F Gradient to ABD Gradient
G9r = Eigen.Vector("G9r",9)
JrT_Gr = Jr.T @ G9r
Cl_Gr = Gen.Closure(
    G9r,
    ti_bar,
    ni_bar,
    bi_bar,
    qi,  # Affine Body i
    tj_bar,
    nj_bar,
    bj_bar,
    qj,  # Affine Body j
) 
# from F Hessian to ABD Hessian
Hr9x9 = Eigen.Matrix("H9x9",9,9)
JrT_Hr_Jr = Jr.T @ Hr9x9 @ Jr
Cl_Hr = Gen.Closure(
    Hr9x9,
    ti_bar,
    ni_bar,
    bi_bar,
    qi,  # Affine Body i
    tj_bar,
    nj_bar,
    bj_bar,
    qj,  # Affine Body j
) 

content += f""" 
// Fixed Joint: C0 C1 C2
// Mapping between ABD qi qj to Fr

{Cl_Fr("Fr", Fr_q)}
{Cl_Gr("JrT_Gr", JrT_Gr)}
{Cl_Hr("JrT_Hr_Jr", JrT_Hr_Jr)}

"""

Fr = Eigen.Vector("Fr", 9)
tij = Fr[0:3, 0]
nij = Fr[3:6, 0]
bij = Fr[6:9, 0]


E_r = kappa * (tij.dot(tij) + nij.dot(nij) + bij.dot(bij))/2

# E01 = 0
Cl_Er = Gen.Closure(kappa, Fr)


dEdFr = VecDiff(E_r, Fr)
ddEdFr = VecDiff(dEdFr, Fr)

content += f""" // Fixed Joint Rotation Energy and Derivatives

{Cl_Er("Er", E_r)}
{Cl_Er("dEdFr", dEdFr)}
{Cl_Er("ddEdFr", ddEdFr)}

// -------------------------------------------------------------------------
"""

$$
C_3 = ||\mathbf{c}_i - \mathbf{c}_j ||_2= \mathbf{0} 
$$

$$
\mathbf{F}_{t} = \begin{bmatrix}
\mathbf{c}_i - \mathbf{c}_j\\
\end{bmatrix}_{3 \times 1}
$$

Frame Affine Body Mapping:

$$
\mathbf{J}_{t} = \begin{bmatrix}
\mathbf{J}(\bar{\mathbf{c}}_i) & -\mathbf{J}(\bar{\mathbf{c}}_j) 
\end{bmatrix}_{3 \times 24}
$$

$$
\mathbf{F}_{t} = \mathbf{J}_{t} \cdot
\begin{bmatrix}
\mathbf{q}_i \\
\mathbf{q}_j \\
\end{bmatrix}
$$

In [58]:
Jt = sp.Matrix.zeros(3,24)
Jt[0:3, 0:12] = compute_J_point(ci_bar)
Jt[0:3, 12:24] = -compute_J_point(cj_bar)


# from ABD q to F
Ft = Jt @ sp.Matrix.vstack(qi, qj)
Cl_Ft = Gen.Closure(
    ci_bar,
    qi,  # Affine Body i
    cj_bar,
    qj,
)  # Affine Body j
# from F Gradient to ABD Gradient
Gt = Eigen.Vector("Gt3", 3)
JtT_Gt = Jt.T @ Gt
Cl_Gt = Gen.Closure(
    Gt,
    ci_bar,
    cj_bar,
)  
# from F Hessian to ABD Hessian
Ht3x3 = Eigen.Matrix("Ht3x3", 3, 3)
JtT_Ht_Jt = Jt.T @ Ht3x3 @ Jt
Cl_Ht = Gen.Closure(
    Ht3x3,
    ci_bar,
    cj_bar,
) 
content += f""" 
// Fixed Joint: C3 C4
// Mapping between ABD qi qj to Ft

{Cl_Ft("Ft", Ft)}
{Cl_Gt("JtT_Gt", JtT_Gt)}
{Cl_Ht("JtT_Ht_Jt", JtT_Ht_Jt)}

"""

Ft = Eigen.Vector("Ft", 3)

Cij = Ft[0:3, 0]


C3 = sqrt(Cij.T * Cij)
Et = 0.5 * kappa * C3**2

Cl_Et = Gen.Closure(kappa, Ft)


dEdFt = VecDiff(Et, Ft)
ddEdFt = VecDiff(dEdFt, Ft)

content += f""" // Fixed Joint Translation Energy and Derivatives

{Cl_Et("Et", Et)}
{Cl_Et("dEdFt", dEdFt)}
{Cl_Et("ddEdFt", ddEdFt)}

// -------------------------------------------------------------------------
"""

In [59]:
path = backend_source_dir("cuda") / "affine_body/constitutions/sym" / "affine_body_fixed_joint.inl"
f = open(path, "w")
f.write(content)
f.close()
print(f"Written to {path}")

Written to /home/ligo/Project/uipc/libuipc/src/backends/cuda/affine_body/constitutions/sym/affine_body_fixed_joint.inl
